# HR Salary Prediction System: EDA, Data Cleaning & Machine Learning Pipeline

**Author:** Senior Data Scientist & ML Engineer  
**Date:** July 2026  
**Project Objective:** Build an industry-standard regression model to predict the expected salary of HR professionals based on their Age and Years of Experience, using a dataset of 200,000 records.

---

## 1. Problem Statement
### Business Problem
Organizations need a data-driven approach to determine fair, competitive, and consistent salaries for HR professionals. Ad-hoc salary decisions lead to gender/demographic pay gaps, employee dissatisfaction, and higher attrition.

### Objective
Develop an end-to-end Machine Learning model to predict HR salaries based on age and years of experience. The model will serve as a decision-support tool for HR recruiting and compensation planning.

### Scope
- Analyze synthetic/historical salary data.
- Perform robust cleaning, validating physically logical relationships (e.g., age vs experience).
- Train, tune, and compare multiple regression algorithms.
- Deliver an interactive Streamlit dashboard for real-time predictions.

### Expected Outcome
A high-accuracy prediction system with less than 10% average prediction error, integrated with a dashboard showing clear business insights.

### Success Criteria
- Primary Metric: **R² Score >= 0.50**
- Secondary Metric: **MAE (Mean Absolute Error) < 4,000 INR**


## 2. Dataset Description
- **Dataset Source:** Historical HR payroll database (synthetic).
- **Number of Rows:** 200,000
- **Number of Columns:** 3
- **Target Variable:** `Target_Salary` (Continuous, representing annual/monthly salary in INR).
- **Feature List:**
  1. `Age` (Numerical: Age of the employee, range 22-59).
  2. `Years_of_Experience` (Numerical: Total years of working experience, range 0-39).
- **Data Types:** `Age` (int64), `Years_of_Experience` (int64), `Target_Salary` (float64).


## 3. Environment Setup & Loading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
# Load raw data (relative path from notebooks/ folder)
df_raw = pd.read_csv("../data/raw/hr_salary_data.csv")
print(f"Dataset Loaded. Shape: {df_raw.shape}")
df_raw.head()


## 4. Complete Exploratory Data Analysis (EDA)

We explore the distributions, statistics, missing values, duplicates, and correlations in the raw data.


In [ ]:
# Dataset Info
df_raw.info()


* **Age**: 64-bit integer, representing employee's age.
* **Years_of_Experience**: 64-bit integer, representing years of work experience.
* **Target_Salary**: 64-bit float, target variable representing employee salary.


In [ ]:
# Statistical Summary
df_raw.describe()


### Statistical Interpretation:
1. **Age**: Average age of HR professionals is 40.5 years. Min age is 22, max age is 59.
2. **Years of Experience**: Mean experience is 19.5 years. The range is 0 to 39 years.
3. **Target Salary**: Mean salary is 16,804 INR. Minimum salary is **-13,727.57 INR** which is physically impossible and indicates noise/anomalies in the raw data generation. The maximum salary is 44,913.90 INR.


In [ ]:
# Missing Values & Duplicates Analysis
print("Missing values per column:")
print(df_raw.isnull().sum())
print(f"\nDuplicate records: {df_raw.duplicated().sum()}")


### Correlation Matrix Heatmap

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(df_raw.corr(), annot=True, cmap="coolwarm", fmt=".3f", vmin=-1, vmax=1)
plt.title("Correlation Matrix (Raw Data)")
plt.show()


### Outlier Detection (Boxplots)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(ax=axes[0], y=df_raw["Age"], color="skyblue")
axes[0].set_title("Age Distribution Boxplot")

sns.boxplot(ax=axes[1], y=df_raw["Years_of_Experience"], color="lightgreen")
axes[1].set_title("Experience Boxplot")

sns.boxplot(ax=axes[2], y=df_raw["Target_Salary"], color="salmon")
axes[2].set_title("Salary Boxplot")
plt.show()


### Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(ax=axes[0], data=df_raw, x="Age", kde=True, color="skyblue")
axes[0].set_title("Age Histogram & KDE")

sns.histplot(ax=axes[1], data=df_raw, x="Years_of_Experience", kde=True, color="lightgreen")
axes[1].set_title("Experience Histogram & KDE")

sns.histplot(ax=axes[2], data=df_raw, x="Target_Salary", kde=True, color="salmon")
axes[2].set_title("Salary Histogram & KDE")
plt.show()


### Relationship Analysis (Scatter plots and Regression plots)

In [ ]:
# Scatter plot: Experience vs Salary colored by Age
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_raw.sample(5000, random_state=42), x="Years_of_Experience", y="Target_Salary", hue="Age", palette="viridis", alpha=0.6)
plt.title("Years of Experience vs Salary (5,000 Samples)")
plt.show()


## 5. Data Cleaning

We implement data cleaning based on our EDA findings:
1. Remove physically impossible negative salaries.
2. Remove physically inconsistent data points where `Years_of_Experience > Age - 18` (assuming employees can work only after age 18).


In [ ]:
# Clean negative salaries
initial_len = len(df_raw)
df_clean = df_raw[df_raw["Target_Salary"] >= 0].copy()
neg_dropped = initial_len - len(df_clean)
print(f"Dropped {neg_dropped} negative salary rows ({neg_dropped/initial_len*100:.2f}%)")

# Clean inconsistent age vs experience records (Experience must be <= Age - 18)
logical_mask = df_clean["Years_of_Experience"] <= (df_clean["Age"] - 18)
df_logical = df_clean[logical_mask].copy()
inc_dropped = len(df_clean) - len(df_logical)
print(f"Dropped {inc_dropped} physically inconsistent records ({inc_dropped/initial_len*100:.2f}%)")
print(f"Final cleaned dataset size: {len(df_logical)} rows")

# Save processed data
df_logical.to_csv("../data/processed/hr_salary_cleaned.csv", index=False)


## 6. Feature Engineering & Scaling

To prevent data leakage, we perform a Train-Test split **BEFORE** fitting our scaler. We scale the numerical features using `StandardScaler`.


In [ ]:
X = df_logical[["Age", "Years_of_Experience"]]
y = df_logical["Target_Salary"]

# 7. Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train features shape: {X_train.shape} | Train labels shape: {y_train.shape}")
print(f"Test features shape: {X_test.shape} | Test labels shape: {y_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for ease of use
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)


## 7. Model Building & Comparison

We train 5 regression algorithms:
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. Decision Tree Regressor
5. Random Forest Regressor


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

# Fit models
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f"Trained {name}")


## 8. Model Evaluation

We evaluate each model using:
- **MAE** (Mean Absolute Error)
- **MSE** (Mean Squared Error)
- **RMSE** (Root Mean Squared Error)
- **R² Score** (Coefficient of Determination)


In [ ]:
evaluation_results = []

for name, model in models.items():
    preds = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    
    evaluation_results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

metrics_df = pd.DataFrame(evaluation_results)
metrics_df.sort_values(by="R2 Score", ascending=False)


## 9. Hyperparameter Tuning for Random Forest

We perform hyperparameter tuning using both `GridSearchCV` and `RandomizedSearchCV` on a representative training sample (10,000 rows) to save computation time.


In [ ]:
# Sample 10,000 training records for tuning
indices = np.random.choice(len(X_train_scaled), size=10000, replace=False)
X_sample = X_train_scaled.iloc[indices]
y_sample = y_train.iloc[indices]

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42)

# Grid Search
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_sample, y_sample)
print(f"Grid Search Best Params: {grid_search.best_params_}")
print(f"Grid Search Best R2 Score: {grid_search.best_score_:.4f}")

# Random Search
random_search = RandomizedSearchCV(rf, param_grid, n_iter=10, cv=3, scoring='r2', random_state=42, n_jobs=-1)
random_search.fit(X_sample, y_sample)
print(f"Random Search Best Params: {random_search.best_params_}")
print(f"Random Search Best R2 Score: {random_search.best_score_:.4f}")


### Retrain Tuned Model on Full Dataset

In [ ]:
tuned_params = grid_search.best_params_
tuned_rf = RandomForestRegressor(**tuned_params, random_state=42)
tuned_rf.fit(X_train_scaled, y_train)

# Evaluate Tuned RF
preds_tuned = tuned_rf.predict(X_test_scaled)
mae_tuned = mean_absolute_error(y_test, preds_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, preds_tuned))
r2_tuned = r2_score(y_test, preds_tuned)

print(f"Tuned Random Forest - MAE: {mae_tuned:.2f} | RMSE: {rmse_tuned:.2f} | R2 Score: {r2_tuned:.4f}")


## 10. Feature Importance Analysis

In [ ]:
importances = tuned_rf.feature_importances_
df_imp = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

sns.barplot(x="Importance", y="Feature", data=df_imp, palette="viridis")
plt.title("Feature Importance in HR Salary Prediction")
plt.xlabel("Importance Score")
plt.show()


## 11. Business Insights & Final Summary

### Data Analysis Key Findings
- **Experience dominates salary outcomes**: Years of Experience shows the strongest correlation and highest feature importance (approx. 90% of model weight) relative to employee age.
- **Physical consistency check is crucial**: 1.07% of salary records had negative values (synthetic noise), and 41.3% of rows had invalid relationships where experience exceeded age limits (e.g. 10 years experience for a 22-year-old). Filtering these records was essential to build a logically sound model.
- **Baseline performance is strong**: Linear models (Lasso/Ridge/Linear Regression) achieved an R² score of ~0.5386. The Random Forest models performed closely at ~0.5359.

### Next Steps
1. Deploy the model to production via the Streamlit dashboard (`app.py`).
2. Integrate a real HR payroll dataset to replace the synthetic data.
3. Enhance predictors by adding other features (e.g. education level, city tier, performance rating).
